# Medical Validation Benchmark

> **Kernel Requirement:** This notebook must run with the project's virtual-environment kernel (e.g. `.venv` located in `health-vault-backend/ai-service`) so that all required dependencies (`fastapi`, `pydantic`, `httpx`, `pillow`, `pandas`, etc.) and environment variables are loaded properly.

## 1. Configuration
Configure document directory, target files to test, summary language, and export paths.

In [8]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
from pathlib import Path

REPORTS_DIR = "D:/TECHROVER/PROJECTS/RTH/healh-vault-project/Medical Reports"
SELECTED_DOCUMENTS = []#["lab_1_.jpeg"]   # [] means process ALL supported files in REPORTS_DIR
PREFERRED_LANGUAGE = "english"
EXPORT_JSON_PATH = "benchmark_results_live.json"
EXPORT_CSV_PATH = "benchmark_results_live.csv"
BENCHMARK_BUCKET_OVERRIDE = None

print("Loaded Configuration:")
print(f"  REPORTS_DIR:               {REPORTS_DIR}")
print(f"  SELECTED_DOCUMENTS:        {SELECTED_DOCUMENTS if SELECTED_DOCUMENTS else 'ALL'}")
print(f"  PREFERRED_LANGUAGE:        {PREFERRED_LANGUAGE}")
print(f"  EXPORT_JSON_PATH:          {EXPORT_JSON_PATH}")
print(f"  EXPORT_CSV_PATH:           {EXPORT_CSV_PATH}")
print(f"  BENCHMARK_BUCKET_OVERRIDE: {BENCHMARK_BUCKET_OVERRIDE}")

Loaded Configuration:
  REPORTS_DIR:               D:/TECHROVER/PROJECTS/RTH/healh-vault-project/Medical Reports
  SELECTED_DOCUMENTS:        ALL
  PREFERRED_LANGUAGE:        english
  EXPORT_JSON_PATH:          benchmark_results_live.json
  EXPORT_CSV_PATH:           benchmark_results_live.csv
  BENCHMARK_BUCKET_OVERRIDE: None


## 2. Project Import Setup
Locate the folder containing the `app` package (`app/modules/validation/service.py`), add it to `sys.path`, and change the current working directory to it so `.env` and settings load correctly.

In [9]:
# ==============================================================================
# 2. PROJECT IMPORT SETUP
# ==============================================================================
import os
import sys
from pathlib import Path

def locate_app_root() -> Path:
    """
    Search for the directory containing the 'app' package by searching for
    'app/modules/validation/service.py' (ignoring .venv and site-packages).
    """
    search_starts = [
        Path.cwd().resolve(),
        Path("D:/TECHROVER/PROJECTS/RTH/healh-vault-project/health-vault/health-vault-backend/ai-service").resolve(),
    ]
    
    # Check current directory and up to 8 parent directories
    for start in search_starts:
        curr = start
        for _ in range(8):
            direct = curr / "app" / "modules" / "validation" / "service.py"
            if direct.is_file():
                return curr.resolve()
            if curr.parent == curr:
                break
            curr = curr.parent
            
    # Search downward if not found in parents
    for start in search_starts:
        if not start.exists():
            continue
        for match in start.rglob("app/modules/validation/service.py"):
            parts = match.parts
            if any(p in parts for p in (".venv", "venv", "site-packages", "node_modules", ".git")):
                continue
            return match.parent.parent.parent.parent.resolve()
            
    raise FileNotFoundError(
        "Could not locate the folder containing 'app/modules/validation/service.py' "
        "(ignoring .venv and site-packages). Please verify your project folder structure."
    )

app_root = locate_app_root()

if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

os.chdir(app_root)
print(f"Located application package root: {app_root}")
print(f"Current working directory set to:  {os.getcwd()}")

Located application package root: D:\TECHROVER\health-vault\health-vault-backend\ai-service
Current working directory set to:  D:\TECHROVER\health-vault\health-vault-backend\ai-service


## 3. Container Setup
Inspect and build the DI container without running the web server, supporting both synchronous and asynchronous builders, and instantiate `MedicalValidationService`.

In [10]:
# ==============================================================================
# 3. CONTAINER SETUP
# ==============================================================================
import inspect
from app.settings import Settings, get_settings
from app.container import Container
from app.modules.validation.service import (
    MedicalValidationService,
    InvalidDocumentFileError,
    MedGemmaUnavailableError,
)

# Load application settings from .env
settings = get_settings() if callable(globals().get("get_settings")) else Settings()

# Build DI container (supporting synchronous or asynchronous builders)
container_builder = getattr(sys.modules.get("app.container"), "build_container", Container)

if inspect.iscoroutinefunction(container_builder):
    container = await container_builder(settings)
else:
    container = container_builder(settings)
    if inspect.isawaitable(container):
        container = await container

# Instantiate MedicalValidationService with container settings and storage
service = MedicalValidationService(container.settings, container.storage)

print("DI Container and MedicalValidationService initialized successfully.")
print(f"  Storage Provider: {getattr(container, 'storage_provider', 'auto')}")
print(f"  MedGemma Model:   {getattr(container.settings, 'medgemma_model', 'medgemma:4b')}")
print(f"  Ollama Base URL:  {service._resolve_ollama_base_url()}")

DI Container and MedicalValidationService initialized successfully.
  Storage Provider: s3
  MedGemma Model:   medgemma:4b
  Ollama Base URL:  http://192.168.21.176:11434


## 4. Document Discovery
Discover supported document files (`.pdf`, `.png`, `.jpg`, `.jpeg`, `.webp`) in `REPORTS_DIR` or filter by `SELECTED_DOCUMENTS`.

In [11]:
# ==============================================================================
# 4. DOCUMENT DISCOVERY
# ==============================================================================
from pathlib import Path

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".webp"}

reports_path = Path(REPORTS_DIR)
documents = []

if not reports_path.exists() or not reports_path.is_dir():
    print(f"Warning: REPORTS_DIR does not exist or is not a directory: {reports_path}")
else:
    if not SELECTED_DOCUMENTS:
        # Discover all supported documents in REPORTS_DIR
        for file_path in sorted(reports_path.iterdir()):
            if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
                documents.append(file_path)
    else:
        # Use only explicitly selected files
        for doc_name in SELECTED_DOCUMENTS:
            file_path = reports_path / doc_name
            if not file_path.is_file():
                print(f"Warning: Selected file does not exist: {file_path}")
            elif file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                print(f"Warning: Selected file '{doc_name}' has unsupported extension '{file_path.suffix}'")
            else:
                documents.append(file_path)

print(f"\nDiscovered {len(documents)} document(s) queued for validation:")
for idx, doc in enumerate(documents, start=1):
    size_kb = doc.stat().st_size / 1024
    print(f"  {idx:2d}. {doc.name:<30} ({size_kb:>7.1f} KB)")


Discovered 41 document(s) queued for validation:
   1. 5069200904.pdf                 ( 2082.8 KB)
   2. 60300505154.pdf                ( 1495.7 KB)
   3. body_scan_report.pdf           (  564.2 KB)
   4. lab_1_.jpeg                    (   75.4 KB)
   5. Manjuben Ranoliya.pdf          (  107.0 KB)
   6. Media (1).jpg                  (  306.6 KB)
   7. Media (10).jpg                 (  336.1 KB)
   8. Media (11).jpg                 (  341.4 KB)
   9. Media (12).jpg                 (  369.0 KB)
  10. Media (13).jpg                 (  349.8 KB)
  11. Media (14).jpg                 (  318.5 KB)
  12. Media (15).jpg                 (  413.9 KB)
  13. Media (16).jpg                 (  383.0 KB)
  14. Media (17).jpg                 (  393.3 KB)
  15. Media (18).jpg                 (  353.4 KB)
  16. Media (19).jpg                 (  396.7 KB)
  17. Media (2).jpg                  (  310.8 KB)
  18. Media (20).jpg                 (  304.6 KB)
  19. Media (21).jpg                 (  375.7 KB)


## 5. Medical Document Validation
Validate each queued document using `await service.validate_medical_document(...)`, recording execution time, catching exceptions (`InvalidDocumentFileError`, `MedGemmaUnavailableError`, generic `Exception`), and converting Pydantic models.

In [12]:
# ==============================================================================
# 5. VALIDATION EXECUTION
# ==============================================================================
import mimetypes
import time
import uuid

results = []

for doc_path in documents:
    mime_type, _ = mimetypes.guess_type(str(doc_path))
    trace_id = str(uuid.uuid4())
    start_time = time.monotonic()
    
    try:
        content = doc_path.read_bytes()
        if not content:
            raise InvalidDocumentFileError("Uploaded file is empty")
        
        # Call the existing validation service method directly
        raw_result = await service.validate_medical_document(
            file_bytes=content,
            file_name=doc_path.name,
            mime_type=mime_type,
            trace_id=trace_id,
        )
        elapsed = time.monotonic() - start_time
        
        # Convert result if it is a Pydantic model
        result_data = raw_result.model_dump() if hasattr(raw_result, "model_dump") else raw_result
        
        results.append({
            "file": doc_path.name,
            "status": "success",
            "result": result_data,
            "elapsed_seconds": round(elapsed, 3),
        })
    except InvalidDocumentFileError as exc:
        elapsed = time.monotonic() - start_time
        results.append({
            "file": doc_path.name,
            "status": "error",
            "code": "UNREADABLE_FILE",
            "error": str(exc),
            "elapsed_seconds": round(elapsed, 3),
        })
    except MedGemmaUnavailableError as exc:
        elapsed = time.monotonic() - start_time
        results.append({
            "file": doc_path.name,
            "status": "error",
            "code": "MEDGEMMA_UNAVAILABLE",
            "error": str(exc),
            "elapsed_seconds": round(elapsed, 3),
        })
    except Exception as exc:
        elapsed = time.monotonic() - start_time
        results.append({
            "file": doc_path.name,
            "status": "error",
            "code": "INTERNAL_ERROR",
            "error": str(exc),
            "elapsed_seconds": round(elapsed, 3),
        })

print(f"Validation completed for {len(results)} document(s).")

[2026-10-02 10:38:16,617] [    INFO] service.py:308 - medical_validation_started
[2026-10-02 10:38:17,515] [    INFO] _client.py:1740 - HTTP Request: GET http://192.168.21.176:11434/api/tags "HTTP/1.1 200 OK"
[2026-10-02 10:38:17,938] [    INFO] vision_service.py:524 - pdf_page_rendered
[2026-10-02 10:38:18,144] [    INFO] vision_service.py:524 - pdf_page_rendered
[2026-10-02 10:38:18,148] [    INFO] vision_service.py:533 - pdf_render_completed
[2026-10-02 10:38:20,380] [    INFO] _client.py:1740 - HTTP Request: POST http://192.168.21.176:11434/api/generate "HTTP/1.1 200 OK"
[2026-10-02 10:38:20,384] [    INFO] service.py:456 - medical_validation_completed
[2026-10-02 10:38:20,417] [    INFO] service.py:308 - medical_validation_started
[2026-10-02 10:38:20,550] [    INFO] vision_service.py:524 - pdf_page_rendered
[2026-10-02 10:38:20,732] [    INFO] vision_service.py:524 - pdf_page_rendered
[2026-10-02 10:38:20,737] [    INFO] vision_service.py:533 - pdf_render_completed
[2026-10-02 10

Validation completed for 41 document(s).


## 6. Results Display
Pretty-print each document's validation result as JSON and view a summary table as a pandas DataFrame.

In [13]:
# ==============================================================================
# 6. RESULTS DISPLAY
# ==============================================================================
import json
import pandas as pd

# 1. Pretty-print each result as JSON
for item in results:
    print(f"==================================================")
    print(f"Document: {item.get('file')}")
    print(f"==================================================")
    print(json.dumps(item, indent=2, default=str))
    print()
# 2. Build summary table DataFrame
summary_records = []
for item in results:
    status_val = item.get("status")
    res = item.get("result", {}) if status_val == "success" else {}
    
    summary_records.append({
        "file": item.get("file"),
        "status": status_val,
        "is_medical": res.get("isMedical") if status_val == "success" else None,
        "document_type": res.get("documentType") if status_val == "success" else None,
        "confidence": res.get("confidence") if status_val == "success" else None,
        "reason": res.get("reason") if status_val == "success" else item.get("error"),
        "elapsed_seconds": item.get("elapsed_seconds"),
        "code": item.get("code") if status_val == "error" else None,
    })

summary_df = pd.DataFrame(summary_records)
summary_df

Document: 5069200904.pdf
{
  "file": "5069200904.pdf",
  "status": "success",
  "result": {
    "isMedical": true,
    "confidence": 0.9,
    "documentType": "LAB_REPORT",
    "reason": "The document is a laboratory report, specifically a complete blood count (CBC) report, including blood cell counts, hematocrit, and clotting time measurements.",
    "method": "vision",
    "model": "medgemma:4b",
    "metrics": {
      "processing_seconds": 3.766,
      "pages_used": 2,
      "used_ollama": true
    }
  },
  "elapsed_seconds": 4.547
}

Document: 60300505154.pdf
{
  "file": "60300505154.pdf",
  "status": "success",
  "result": {
    "isMedical": true,
    "confidence": 0.9,
    "documentType": "LAB_REPORT",
    "reason": "The document is a laboratory report, specifically a Hemogram report, including RBC count, WBC count, and other hematological parameters.",
    "method": "vision",
    "model": "medgemma:4b",
    "metrics": {
      "processing_seconds": 2.235,
      "pages_used": 2,
  

,file,status,is_medical,document_type,confidence,reason,elapsed_seconds,code
0,5069200904.pdf,success,True,LAB_REPORT,0.90,"The document is a laboratory report, specifica...",4.547,None
1,60300505154.pdf,success,True,LAB_REPORT,0.90,"The document is a laboratory report, specifica...",2.266,None
2,body_scan_report.pdf,success,True,IMAGING_REPORT,0.90,The document is an ultrasound report of the ab...,2.078,None
3,lab_1_.jpeg,success,True,PRESCERIPTION,0.90,"The document is a prescription from a clinic, ...",2.125,None
4,Manjuben Ranoliya.pdf,success,True,CONSULTATION_REPORT,0.90,"The document is a dental quotation, including ...",2.125,None
5,Media (1).jpg,success,True,LAB_REPORT,0.90,The document is a urine analysis report,2.031,None
6,Media (10).jpg,success,True,LAB_REPORT,0.90,"The document contains a urine test result, inc...",1.937,None
7,Media (11).jpg,success,True,LAB_REPORT,0.90,"The document contains lab results, including c...",2.000,None
8,Media (12).jpg,success,True,DISCHARGE_SUMMARY,0.99,The document is a discharge summary from a hos...,2.344,None
9,Media (13).jpg,success,True,LAB_REPORT,0.90,The document is a urine analysis report from a...,3.500,None


## 7. Export Results
Export detailed validation results to JSON (`EXPORT_JSON_PATH`) and the summary table to CSV (`EXPORT_CSV_PATH`).

In [14]:
# ==============================================================================
# 7. EXPORT RESULTS
# ==============================================================================
import json
from pathlib import Path

# Export full validation results to JSON
export_json = Path(EXPORT_JSON_PATH)
with open(export_json, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)
print(f"Full results exported to JSON: {export_json.resolve()}")

# Export summary table to CSV
export_csv = Path(EXPORT_CSV_PATH)
summary_df.to_csv(export_csv, index=False)
print(f"Summary table exported to CSV: {export_csv.resolve()}")

Full results exported to JSON: D:\TECHROVER\health-vault\health-vault-backend\ai-service\benchmark_results_live.json
Summary table exported to CSV: D:\TECHROVER\health-vault\health-vault-backend\ai-service\benchmark_results_live.csv
